# Demo - Claude Agent SDK: PreToolUse and PostToolUse hooks

## Setup

Install the Claude Agent SDK and dotenv packages

In [ ]:
!uv pip install claude_agent_sdk dotenv

Import the `claude_agent_sdk` package and a bunch of other imports needed to create hooks.

In [ ]:
import json
from datetime import datetime, timezone
from dotenv import load_dotenv, find_dotenv

# Import your Anthropic API key - ensure you have ANTHROPIC_API_KEY define in an .env file
load_dotenv(find_dotenv())

from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    ClaudeSDKClient,
    HookMatcher,
    TextBlock,
    ToolUseBlock,
    create_sdk_mcp_server,
    tool,
)

## Constants and helpers

Some constants to represent the status of an order (the use case in this demo) and a helper function to convert Unix timestamps (epoch) into an ISO DateTime object

In [ ]:
STATUS = {1: "pending", 2: "packed", 3: "shipped", 4: "delivered"}


def to_iso(epoch):
    return datetime.fromtimestamp(epoch, tz=timezone.utc).date().isoformat()

## Tools

Define two tools which we will use in this demo:
* `lookup_order` which will lookup an order
* `process_refund` which will process the refund for an order

We will use the Claude Agent SDK's in-process MCP server to make these tools available to our agent. 

In [ ]:
# Lookup order tool
@tool("lookup_order",
      "Look up an order. Use when an order number (like #12345) appears. "
      "Input: order_id.",
      {"order_id": str})
async def lookup_order(args):
    raw = {"date": 1781136000, "status": 3}          # epoch + numeric code
    return {"content": [{"type": "text", "text": json.dumps(raw)}]}

# Process refund tool
@tool("process_refund",
      "Process a refund for an order. Inputs: order_id, amount in dollars.",
      {"order_id": str, "amount": float})
async def process_refund(args):
    return {"content": [{"type": "text",
                         "text": json.dumps({"approved": True, **args})}]}

# Create the MCP server and register the tools
ORDERS = create_sdk_mcp_server(name="orders", version="1.0.0",
                               tools=[lookup_order, process_refund])

## Hooks

We define two hooks:
* `refund_gate` executes before tool use, and will block calls to `process_refund` if the amount is greater than $500 (as for refunds about $500 we might, for example, have a business requirement which requires human approval)

* `normalise` executes after tool use: converts a date in Unix timestamp format to a DateTime object, and converts a numerical order status to a string value - both use the helper functions/constants defined previously

In [ ]:
async def refund_gate(input_data, tool_use_id, context):
    """PreToolUse — gate the CALL before it executes. Deterministic."""
    amount = input_data["tool_input"].get("amount", 0)
    if amount > 500:
        print(f"    PRE  BLOCKED  process_refund({input_data['tool_input']})"
              f"  amount > $500")
        print( "         permissionDecision: deny  "
               "(the reason travels to the model)")
        return {"hookSpecificOutput": {
            "hookEventName": "PreToolUse",
            "permissionDecision": "deny",
            "permissionDecisionReason":
                "Refunds over $500 require a human agent. Escalate this "
                "case and reassure the customer politely.",
        }}
    return {}


async def normalise(input_data, tool_use_id, context):
    """PostToolUse — REPLACE the result before the model sees it."""
    resp = input_data["tool_response"]
    blocks = resp if isinstance(resp, list) else resp.get("content", [])
    for b in blocks:
        if b.get("type") == "text":
            raw = json.loads(b["text"])
            fixed = {"date": to_iso(raw["date"]),        # 1781136000 -> 2026-06-11
                     "status": STATUS[raw["status"]]}    # 3 -> "shipped"
            print(f"    POST before: {raw}")
            print(f"    POST after:  {fixed}")
            new_blocks = [{"type": "text", "text": json.dumps(fixed)}]
            updated = (new_blocks if isinstance(resp, list)
                       else {**resp, "content": new_blocks})
            return {"hookSpecificOutput": {
                "hookEventName": "PostToolUse",
                "updatedToolOutput": updated,
            }}
    return {}

## Setup the Claude Agent

In [ ]:
OPTIONS = ClaudeAgentOptions(
    mcp_servers={"orders": ORDERS},
    allowed_tools=["mcp__orders__lookup_order",
                   "mcp__orders__process_refund"],
    system_prompt=("You are a customer support agent. Be brief. Order details "
                   "have already been verified by the front desk, so act on the "
                   "customer's request directly using your tools rather than "
                   "asking follow-up questions or second-guessing the amount."),
    max_turns=4,
    hooks={
        "PreToolUse":  [HookMatcher(matcher="mcp__orders__process_refund",
                                    hooks=[refund_gate])],
        "PostToolUse": [HookMatcher(matcher="mcp__orders__lookup_order",
                                    hooks=[normalise])],
    },
)

async def run_task(label, prompt):
    print(f"\n== {label} ==")
    print(f'user: "{prompt}"')
    async with ClaudeSDKClient(options=OPTIONS) as client:
        await client.query(prompt)
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, ToolUseBlock):
                        print(f"  model called {block.name}({block.input})")
                    elif isinstance(block, TextBlock) and block.text.strip():
                        print(f'model: "{block.text.strip()}"')

## Run the agent with hooks

In [ ]:
async def main():
    await run_task("Task 1: PostToolUse replaces what comes back",
                   "When did order #12345 ship? Give me the date.")
    await run_task("Task 2: PreToolUse gates what goes out",
                   "Please process a $650 refund for order #12345.")


await main()